# 4.0 — Pydantic Models over the Neo4j Graph

Exercises the schemas in `src/data/schemas/` fed from the pharmacovigilance
graph in **Neo4j** (SQLite→Neo4j migration). The bridge already exists in
`src/data/enrichment.py`; here we only use and validate it.

If Neo4j is unreachable or unauthenticated, the notebook **skips** the DB
cells with a message and finishes without error.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent  # the notebook lives in notebooks/ -> repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from datetime import date

from neo4j.exceptions import AuthError, ServiceUnavailable

from src.config import neo4j_config
from src.data.enrichment import PharmacovigilanceStore, enrich_drug
from src.data.repository import get_enriched_drug
from src.data.schemas import ATCCode, Drug, Interaction, Med, Patient, SideEffect

from pprint import pprint, pformat

cfg = neo4j_config()
print(f"uri={cfg.uri}  user={cfg.user}  database={cfg.database}")  # never the password

uri=bolt://localhost:7687  user=neo4j  database=neo4j


## Connection and CID discovery

Opens the `PharmacovigilanceStore`. On a connection or auth failure, `STORE =
None` and the following cells are skipped. When connected, it looks for a
`:Drug` CID with at least one `HAS_SIDE_EFFECT` edge (prefers 33613,
amoxicillin; otherwise the first one found).

In [ ]:
STORE: PharmacovigilanceStore | None = None
SAMPLE_CID: int | None = None

# Discover a Drug node with at least one side-effect edge, returning the
# whole node so we can show every label and property (not just the CID).
#_DISCOVER = (
#    "MATCH (d:Drug)-[:HAS_SIDE_EFFECT]->() "
#    "WITH DISTINCT d ORDER BY (d.cid = 33613) DESC, d.cid LIMIT 1 "
#    "RETURN d.cid AS cid, labels(d) AS labels, properties(d) AS props"
#)

# Optional (testing): pick a RANDOM drug that has a side-effect edge, over the
# whole set of drugs, instead of the deterministic 33613-preferred pick above.
# Uncomment to override _DISCOVER (rand() randomizes server-side; same columns).
_DISCOVER = (
    "MATCH (d:Drug)-[:HAS_SIDE_EFFECT]->() "
    "WITH DISTINCT d ORDER BY rand() LIMIT 1 "
    "RETURN d.cid AS cid, labels(d) AS labels, properties(d) AS props"
)

try:
    STORE = PharmacovigilanceStore(cfg)
    with STORE._driver.session(database=cfg.database) as _s:
        _rec = _s.run(_DISCOVER).single()
    SAMPLE_CID = _rec["cid"] if _rec else None
    if SAMPLE_CID is None:
        print("Neo4j connected but no HAS_SIDE_EFFECT edges — skipping the DB.")
    else:
        print(f"Connected. Sample CID = {SAMPLE_CID}")
        print(f"  labels: {_rec['labels']}")
        for _key, _value in _rec["props"].items():
            print(f"  {_key}: {_value}")
except (ServiceUnavailable, AuthError) as exc:
    if STORE is not None:
        STORE.close()
    STORE = None
    print(f"Neo4j unavailable ({type(exc).__name__}) — skipping DB cells.")

Neo4j unavailable (AuthError) — skipping DB cells.


## Graph rows → validated models

`side_effects` / `interactions` return already-validated Pydantic models
(mandatory provenance, pattern-checked MedDRA code, severity and source
literals). We check the type and show the first few.

In [3]:
if STORE is None or SAMPLE_CID is None:
    print("DB skipped — no effects/interactions to show.")
else:
    effects = STORE.side_effects(SAMPLE_CID)
    interactions = STORE.interactions(SAMPLE_CID)
    assert all(isinstance(e, SideEffect) for e in effects)
    assert all(isinstance(i, Interaction) for i in interactions)
    print(f"CID {SAMPLE_CID}: {len(effects)} effects, {len(interactions)} interactions")
    for e in effects[:10]:
        pprint(e.model_dump())
    for i in interactions[:10]:
        pprint(i.model_dump())

DB skipped — no effects/interactions to show.


## Enrich a local Drug (no network)

We build a `Drug` by hand and enrich it from the graph with `enrich_drug` —
the primary path, with no dependency on PubChem.

In [4]:
enriched: Drug | None = None
if STORE is None or SAMPLE_CID is None:
    print("DB skipped — Drug not enriched.")
else:
    base = Drug(cid=SAMPLE_CID, name=f"CID {SAMPLE_CID}")
    enriched = enrich_drug(base, STORE)
    print(f"has_atc={enriched.has_atc}  "
          f"effects={len(enriched.side_effects)}  "
          f"interactions={len(enriched.interactions)}")

DB skipped — Drug not enriched.


## Full patient from the graph

We wrap the enriched `Drug` in a `Med` and a `Patient` (same shape as
notebook 2.0). This demonstrates that graph-fed models compose the full
domain hierarchy and still pass every validator.

In [5]:
if enriched is None:
    print("DB skipped — Patient not built.")
else:
    med = Med(
        ATC_code=(enriched.chemical_group if enriched.has_atc else ATCCode(code="J01CA04")),
        name="Test medication",
        dosage="500 mg",
        frequency="every 8 hours",
        start_date=date(2026, 6, 1),
        active_principles=[enriched],
    )
    patient = Patient(
        id=1,
        name="Test patient",
        age=78,
        birth_date=date(1948, 1, 1),
        number_of_meds=1,
        polymedicated=True,
        diseases=["respiratory infection"],
        medication=[med],
    )
    print(patient.model_dump())

DB skipped — Patient not built.


## Optional — PubChem + graph ⚠️

`get_enriched_drug` resolves chemical identity via **PubChem (network)** and
adds pharmacovigilance data from the graph. It is not required to validate
the notebook.

In [6]:
if STORE is None or SAMPLE_CID is None:
    print("DB skipped — PubChem not queried.")
else:
    full = get_enriched_drug(SAMPLE_CID, STORE)
    print(full.model_dump() if full else "CID not resolved in PubChem")

DB skipped — PubChem not queried.


## Wrap-up

In [7]:
if STORE is not None:
    STORE.close()
    print("Store closed.")
else:
    print("No store was open.")

No store was open.
